# Retrieval with a Feedback Loop

Learn from user ratings: boost chunks that helped on similar past questions, demote those that did not.

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
FEEDBACK_FILE = DATA / "feedback.json"
feedback = json.loads(FEEDBACK_FILE.read_text()) if FEEDBACK_FILE.exists() else []

def record_feedback(query, chunk_idx, rating):
    """rating: +1 helpful, -1 not helpful"""
    feedback.append({"query": query, "chunk": chunk_idx, "rating": rating})
    FEEDBACK_FILE.write_text(json.dumps(feedback, indent=2))

At query time, each past rating is weighted by how similar its question is to the new one.

In [ ]:
import numpy as np

def search_with_feedback(query, k=3, boost=0.15):
    scores = store.vecs @ embed(query)[0]
    if feedback:
        past = embed([f["query"] for f in feedback]) @ embed(query)[0]
        for f, sim in zip(feedback, past):
            if sim > 0.5:
                scores[f["chunk"]] += boost * f["rating"] * sim
    return [{"idx": int(i), "text": chunks[i], "score": float(scores[i]), "meta": {}} for i in np.argsort(-scores)[:k]]

query = "How do I measure the quality of retrieval?"
before = search_with_feedback(query)
show(before, width=90)

Simulate a user marking the top hit as unhelpful and the third as helpful, then ask a *similar* question.

In [ ]:
record_feedback(query, before[0]["idx"], -1)
record_feedback(query, before[2]["idx"], +1)
print("--- after feedback, similar query")
show(search_with_feedback("How can retrieval quality be evaluated?"), width=90)

In production, store feedback in a database, decay old ratings and use the signal to fine-tune the retriever.